# Game Bot Creator -- Object Detection Training (YOLO)

Alternative to `train_object_detector.ipynb` (RF-DETR) using Ultralytics YOLO11 instead. Trains a small object-detection model on a dataset captured/labeled by Game Bot Creator's "Train new model..." wizard (Detect Object action / AI tab), and exports it to the exact same `.onnx` contract `onnx_detector.py` already expects -- confirmed directly against that file, not just assumed: gray-114-padded letterbox preprocessing, RGB/255.0/CHW blob, and a graceful squeeze of a leading batch dimension are all already there, written generically enough to cover "every YOLO/RF-DETR-family model" (onnx_detector.py's own words). So either notebook produces a drop-in `best.onnx` for the same "+ Detect Model" action -- no app-side changes needed to use this one instead of, or alongside, the RF-DETR notebook.

**Model:** `yolo11s.pt` (Ultralytics YOLO11, small) -- picked as a balance between detection reliability and training/export speed for a small, single-class dataset. `yolo11n.pt` (nano) is faster but has noticeably less capacity for a harder object (e.g. one that can appear at any rotation); `yolo11m.pt`/`l`/`x` are more accurate on large datasets but much likelier to overfit a captured dataset this small (typically a few dozen frames) and slower to train/export for no real benefit here.

**Colab (default, manual):**
1. `Runtime` -> `Change runtime type` -> select a **GPU** (T4 is fine, free tier).
2. `Runtime` -> `Run all`.
3. Wait for a "Choose files" button to appear below step 2's cell (can take a minute -- it installs dependencies first), click it, and upload the `..._dataset.zip` file Game Bot Creator produced.
4. Everything else runs on its own -- training, export, and an optional sanity check (step 6, safe to skip) -- then `best.onnx` downloads automatically at the end (or copies to Google Drive instead, see step 7). Import it back into the app via the AI tab's "Import Model...".

**Kaggle (opt-in, scripted):** the app's `cv_training.py` pushes a notebook as a Kaggle kernel with the dataset attached as a Kaggle Dataset, polls until it finishes, and pulls `best.onnx` back automatically -- no manual steps. This notebook auto-detects which platform it's running on (`ON_KAGGLE`, step 2) and adjusts the upload/download cells accordingly, same as the RF-DETR notebook.

**No click-point (keypoint) prediction** -- this notebook only trains box detection, matching the app's `[N,6]` (x1,y1,x2,y2,confidence,class_id) contract; `onnx_detector.py` already handles a keypoint-less model by clicking the box center instead. YOLO-pose (`yolo11s-pose.pt`) supports a custom single-keypoint schema and could add this later, mirroring RF-DETR's keypoint-preview mode -- not attempted here, to keep this notebook's first version comparable and simple.

**Status: not yet verified end-to-end against a real training run.** Code-complete against Ultralytics' documented training/export API -- but exactly like the RF-DETR notebook's own history, expect at least one round of real-Colab-run debugging before treating this as fully proven. See `docs/cv-object-detection-investigation.md` in the main game-bot repo for that notebook's iteration history as a preview of the kind of issues that tend to show up only on a real run.

## 1. Install dependencies

Pinned, not "latest" -- same reasoning as the RF-DETR notebook: an unpinned install risks a future upstream release silently changing behavior underneath this notebook. `ultralytics==8.4.137` is the latest stable release as of this notebook's writing and includes YOLO11; the `onnx`/`onnxruntime` pins match the RF-DETR notebook's own (only the ONNX opset needs to line up with what `onnx_detector.py` can load, not the exact training-time package version).

In [ ]:
!pip install -q "ultralytics==8.4.137" "onnx==1.22.0" "onnxruntime==1.29.0"

### Optional: check what hardware this session got

Purely informational -- doesn't change how training runs either way. `BATCH = -1` in step 4 already uses Ultralytics' own AutoBatch to size itself to whatever GPU you're connected to, free tier or a Colab Pro/Pro+ pick.

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print('\nYour runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
if ram_gb < 20:
    print('Not using a high-RAM runtime')
else:
    print('You are using a high-RAM runtime!')

## 2. Upload and unpack the dataset

Expects the zip Game Bot Creator's "Train new model..." wizard produces: an `images/` folder of captured frames plus a `labels.json` of the form
`{"images_dir": "images", "labels": [{"image": "frame_001.png", "objects": [{"box": [x, y, w, h], "keypoint": [x, y]}, ...]}, ...]}`
(frames with no entry in `labels` were skipped during labeling -- the object wasn't visible in them; each frame can list more than one object). The keypoint field is ignored by this notebook -- see the intro above for why.

In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path

RAW_DIR = Path("dataset_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

# cv_training.py's push already uploaded the dataset zip as this kernel's one attached
# Kaggle Dataset -- mounted read-only under /kaggle/input, no interactive upload prompt
# needed the way Colab's files.upload() requires. Detected by whether a zip is actually
# there, not just os.path.exists("/kaggle/input") -- that directory exists (empty) on
# plain Colab too. Recursive, not a fixed single-level "*/labels.json" -- Kaggle actually
# mounts datasets under /kaggle/input/datasets/<owner>/<dataset>/, not a flat
# /kaggle/input/<dataset>/, confirmed directly against a real run (a single-level glob
# always came back empty there, silently falling through to the google.colab.files.upload()
# branch below -- which Kaggle's image really does have installed -- and hanging forever
# waiting for a browser upload dialog that can never appear in a non-interactive kernel run).
candidates = glob.glob("/kaggle/input/**/labels.json", recursive=True)
ON_KAGGLE = bool(candidates)
if ON_KAGGLE:
    shutil.copytree(Path(candidates[0]).parent, RAW_DIR)
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(RAW_DIR)

print("Extracted:", list(RAW_DIR.iterdir()))

## 3. Convert to YOLO format (train/val split)

One class (`"object"`, id `0`). Ultralytics expects `images/{train,val}/*.png` plus one `.txt` label file per image (`labels/{train,val}/*.txt`, same filename stem), each line `class_id cx cy w h` -- center-x/y and width/height, all normalized to [0, 1] by image size. A 90/10 split is a reasonable default for a small, single-object dataset; adjust `VALID_FRACTION` if you captured a lot more images. A frame with zero labeled objects gets an empty `.txt` file -- a real negative example, same convention as the RF-DETR notebook's COCO conversion. If the dataset is too small for even one real validation image, `val` falls back to reusing the training images (Ultralytics needs *some* non-empty `val` split to run its own validation/early-stopping loop at all) -- printed clearly when that happens, since it means the reported validation metrics aren't a genuine held-out check in that case.

In [ ]:
import json, random
import cv2

VALID_FRACTION = 0.1
YOLO_DIR = Path("dataset_yolo")
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

with open(RAW_DIR / "labels.json") as f:
    raw = json.load(f)
images_dir = RAW_DIR / raw.get("images_dir", "images")
entries = raw["labels"]

random.seed(0)
random.shuffle(entries)
n_valid = max(1, int(len(entries) * VALID_FRACTION)) if len(entries) > 1 else 0
splits = {"val": entries[:n_valid], "train": entries[n_valid:]}
if not splits["val"]:
    print("Dataset too small for a real validation split -- reusing the training images for "
          "'val' instead. Ultralytics needs a non-empty val split to run at all, but the "
          "printed validation metrics won't reflect genuine held-out performance until you "
          "have enough frames for VALID_FRACTION to carve out at least one.")
    splits["val"] = splits["train"]

for split_name, split_entries in splits.items():
    (YOLO_DIR / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split_name).mkdir(parents=True, exist_ok=True)
    for e in split_entries:
        src = images_dir / e["image"]
        img = cv2.imread(str(src))
        h, w = img.shape[:2]
        shutil.copy(src, YOLO_DIR / "images" / split_name / e["image"])
        stem = Path(e["image"]).stem
        lines = []
        for obj in e["objects"]:
            bx, by, bw, bh = obj["box"]
            cx, cy = (bx + bw / 2) / w, (by + bh / 2) / h
            nw, nh = bw / w, bh / h
            lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        (YOLO_DIR / "labels" / split_name / f"{stem}.txt").write_text("\n".join(lines))
    print(f"{split_name}: {len(split_entries)} image(s)")

data_yaml = YOLO_DIR / "data.yaml"
data_yaml.write_text(
    f"path: {YOLO_DIR.resolve()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: object\n"
)
print("Wrote", data_yaml)

## 4. Train

`yolo11s.pt` (see intro for why this size) with a conservative augmentation set for a small dataset -- disables everything that would distort the *whole frame* unrealistically (flips/rotation/shear/perspective would mirror or skew the game's own UI, which never happens for real; mosaic/mixup/copy_paste combine multiple images into one composite, the same class of artifact risk the RF-DETR notebook's own copy-paste step turned out to hurt keypoint accuracy with -- left off here until there's a reason to believe it helps box-only detection specifically) and keeps mild color jitter and small translate/scale jitter, the same "start conservative" philosophy as the RF-DETR notebook's `AUG_CONFIG`.

`BATCH = -1` uses Ultralytics' own AutoBatch to size itself to whatever GPU is actually connected (free tier or a Colab Pro/Pro+ pick) -- a long-standing, widely-used Ultralytics feature, unlike RF-DETR's much newer equivalent that turned out to be broken for keypoint models (see the RF-DETR notebook's own investigation). Not yet run against a real dataset through this notebook either, though -- treat it the same way: confirmed as intended in Ultralytics' own docs, not yet confirmed against this exact pipeline.

`PATIENCE = 20` is Ultralytics' own built-in early stopping (same idea, same value, as the RF-DETR notebook's `EARLY_STOPPING_PATIENCE`) -- training stops once validation mAP hasn't improved for this many epochs, instead of always running the full `EPOCHS`.

In [ ]:
from ultralytics import YOLO

EPOCHS = 200      # small dataset -- more epochs than a COCO-scale run, early stopping cuts it short once it plateaus
PATIENCE = 20     # stop once validation mAP hasn't improved for this many epochs
IMGSZ = 640
BATCH = -1        # Ultralytics' own AutoBatch -- sizes itself to whatever GPU is actually connected

model = YOLO("yolo11s.pt")
model.train(
    data=str(YOLO_DIR / "data.yaml"),
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    # Whole-frame geometric augmentation deliberately conservative/off -- see this cell's
    # markdown for why (mirrors/skews the game's own UI unrealistically, or risks the same
    # composite-artifact issue the RF-DETR notebook's copy-paste step ran into).
    fliplr=0.0, flipud=0.0, degrees=0.0, shear=0.0, perspective=0.0,
    mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
    # Mild color/position/scale jitter kept, tuned down from Ultralytics' own defaults for
    # a dataset this small (same "start conservative" reasoning as the RF-DETR notebook's
    # AUG_CONFIG).
    hsv_h=0.015, hsv_s=0.3, hsv_v=0.2, translate=0.1, scale=0.2,
)

## 5. Export to ONNX with the app's expected contract

`nms=True` bakes real IoU-based NMS directly into the exported graph (Ultralytics' own supported feature) -- unlike RF-DETR's export, which only does confidence-based top-k and needed a real NMS pass added on the app side afterward (see `docs/cv-object-detection-investigation.md`'s NMS follow-up). Output shape is `(1, max_det, 6)`: x1, y1, x2, y2, confidence, class_id, in this model's own fixed input-size pixel coordinates -- exactly `onnx_detector.py`'s `[N,6]` contract, plus a leading batch dimension of 1 that `_run_inference` already squeezes off itself (`if outputs.ndim == 3: outputs = outputs[0]`), so no export-side workaround is needed for that either.

`conf=0.001` (not Ultralytics' own default of `0.25`) -- that threshold is baked into the exported graph permanently, discarding anything below it before the model ever produces output. The app's own Detect Model action/Validate preview is what should decide the real confidence cutoff at runtime, not this export step, so this keeps every meaningfully-non-zero candidate in the output instead of silently pruning weak ones the app never gets a chance to see.

Confirmed directly against `onnx_detector.py`, not assumed: its `letterbox()` pads with gray (114,114,114), centered, matching Ultralytics' own preprocessing convention exactly -- and its own docstring already says "the standard fixed-input-size preprocessing every YOLO/RF-DETR-family model expects." No custom export wrapper needed, unlike RF-DETR's (whose high-level `predict()` API wasn't traceable at all) -- `model.export()` alone produces a directly-compatible `best.onnx`.

Exports from `model.trainer.best` (Ultralytics' own best-checkpoint path) rather than the in-memory `model` object straight after `.train()` -- the same "load the best checkpoint explicitly, don't assume the in-memory weights are already it" discipline the RF-DETR notebook needed a real fix for once (see its own "export step traced whatever was left in memory" follow-up).

In [ ]:
best_pt = model.trainer.best  # Ultralytics' own best-checkpoint path, not whatever's left in memory -- see this cell's markdown
onnx_path = YOLO(best_pt).export(
    format="onnx", nms=True, imgsz=IMGSZ, batch=1, opset=17, simplify=True,
    conf=0.001,  # keep weak detections in the output -- see this cell's markdown for why
)
shutil.copy(onnx_path, "best.onnx")
print("Exported best.onnx from", best_pt)

## 6. Sanity-check the export (optional)

Quick check that the exported graph actually produces plausible detections before downloading it -- same idea as the RF-DETR notebook's own sanity-check step. Uses a plain resize rather than a true letterbox for simplicity (same simplification the RF-DETR notebook's own sanity check makes) -- good enough to catch a badly broken export, not meant to exactly reproduce the app's own preprocessing. Safe to skip if you'd rather just download `best.onnx` and test it directly in the app.

In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image

sess = ort.InferenceSession("best.onnx", providers=["CPUExecutionProvider"])
img = Image.open(next((YOLO_DIR / "images" / "val").glob("*.png"))).convert("RGB").resize((IMGSZ, IMGSZ))
blob = (np.array(img).astype(np.float32) / 255.0).transpose(2, 0, 1)[None, ...]
output = sess.run(None, {sess.get_inputs()[0].name: blob})[0]
if output.ndim == 3:
    output = output[0]  # same batch-dim squeeze onnx_detector.py itself already does
print("output shape:", output.shape)  # (max_det, 6): x1,y1,x2,y2,score,label
top = output[np.argsort(-output[:, 4])[:5]]
print(top)

## 7. Download the trained model

**Slow on Colab?** `files.download()` below doesn't do a normal HTTP download -- it base64-encodes the whole file and pushes it through Colab's Python-kernel-to-browser message bridge, which is known to be slow for anything more than a few MB (the exported model usually is). If you've already mounted your Google Drive in this session (`from google.colab import drive; drive.mount('/content/drive')`, a separate cell, one-time permission prompt), set `USE_GOOGLE_DRIVE = True` below to copy `best.onnx` there instead and download it from drive.google.com -- skips the slow bridge entirely.

In [ ]:
if ON_KAGGLE:
    # Kaggle captures every file left in /kaggle/working/ as this kernel's output -- no
    # equivalent of Colab's interactive download; cv_training.py's poll/pull step fetches
    # it afterward via `kaggle kernels output`.
    print("On Kaggle: best.onnx left in /kaggle/working/ -- fetched by the app's Kaggle poll/pull step.")
else:
    USE_GOOGLE_DRIVE = False  # set True if you've already mounted Drive this session -- see this cell's markdown

    if USE_GOOGLE_DRIVE:
        drive_dest = Path("/content/drive/MyDrive/best.onnx")
        if not drive_dest.parent.is_dir():
            raise RuntimeError(
                "Google Drive isn't mounted at /content/drive -- run "
                "`from google.colab import drive; drive.mount('/content/drive')` in a separate cell "
                "first (one-time permission prompt), then re-run this cell."
            )
        shutil.copy("best.onnx", drive_dest)
        print(f"Copied to Google Drive: {drive_dest} -- download it from drive.google.com.")
    else:
        from google.colab import files
        files.download("best.onnx")